# Clasificador de botellas

Este cuaderno usa las fotos que ya están en el repo y deja ver la clasificación encima de la imagen.

Positivas (recorte de botella):

- `img/p`
- `dato/p`
- `dato/img/p2`

Negativas (sin botella):

- `img/n`
- `dato/n`
- `dato/img/n2`

El video de prueba es `video_feed/bottles.mp4`.

El clasificador que ya está entrenado es `botella.xml`, ventana **38×46**, el mismo tamaño que `img/classifier/params.xml`. `app.py` y `detectando.py` lo cargan con `cv2.CascadeClassifier`.

La celda **Ver la clasificación** dibuja las cajas y guarda las fotos en `clasificador/vistazo/`. No hace falta la webcam.

La celda **Compilar** sigue apagada: `opencv_createsamples` y `opencv_traincascade` no vienen con `pip install opencv-python`.

## Así se ve una botella ya marcada

Fotograma de `video_feed/bottles.mp4` con `botella.xml`:

![Botella clasificada](vistazo/video_0000_15.jpg)

In [ ]:
from pathlib import Path
import shutil
import cv2

RAIZ = Path.cwd().resolve()
if RAIZ.name == "clasificador":
    RAIZ = RAIZ.parent

CARPETA = RAIZ / "clasificador"
SALIDA = CARPETA / "salida"
VISTAZO = CARPETA / "vistazo"
INFO = CARPETA / "info.lst"
FONDOS = CARPETA / "bg.txt"
VECTOR = CARPETA / "muestras.vec"
BAT = CARPETA / "compilar.bat"
VIDEO = RAIZ / "video_feed" / "bottles.mp4"

CARPETAS_POSITIVAS = [
    RAIZ / "img" / "p",
    RAIZ / "dato" / "p",
    RAIZ / "dato" / "img" / "p2",
]
CARPETAS_NEGATIVAS = [
    RAIZ / "img" / "n",
    RAIZ / "dato" / "n",
    RAIZ / "dato" / "img" / "n2",
]

ANCHO = 38
ALTO = 46
ETAPAS = 10
EXTENSIONES = {".jpg", ".jpeg", ".png", ".bmp"}

for carpeta in (SALIDA, VISTAZO):
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raíz:", RAIZ)
print("OpenCV:", cv2.__version__)
print("Video:", VIDEO, "existe" if VIDEO.exists() else "no está")

In [ ]:
def listar_fotos(carpeta):
    if not carpeta.is_dir():
        print("No está la carpeta:", carpeta)
        return []
    return sorted(
        ruta for ruta in carpeta.iterdir()
        if ruta.is_file() and ruta.suffix.lower() in EXTENSIONES
    )

def ruta_para_opencv(ruta):
    return ruta.relative_to(RAIZ).as_posix()

positivas = [foto for carpeta in CARPETAS_POSITIVAS for foto in listar_fotos(carpeta)]
negativas = [foto for carpeta in CARPETAS_NEGATIVAS for foto in listar_fotos(carpeta)]

lineas_info = []
for foto in positivas:
    imagen = cv2.imread(str(foto))
    if imagen is None:
        print("No se pudo leer:", foto)
        continue
    alto, ancho = imagen.shape[:2]
    lineas_info.append(f"{ruta_para_opencv(foto)} 1 0 0 {ancho} {alto}")

INFO.write_text("\n".join(lineas_info) + ("\n" if lineas_info else ""), encoding="utf-8")
FONDOS.write_text(
    "\n".join(ruta_para_opencv(foto) for foto in negativas) + ("\n" if negativas else ""),
    encoding="utf-8",
)

print(f"Positivas útiles: {len(lineas_info)}")
print(f"Negativas: {len(negativas)}")
print("Listas escritas:", INFO.name, "y", FONDOS.name)
for carpeta in CARPETAS_POSITIVAS + CARPETAS_NEGATIVAS:
    print(f"  {ruta_para_opencv(carpeta)}: {len(listar_fotos(carpeta))}")

## Ver la clasificación

Carga `botella.xml` y pinta las cajas sobre un fotograma del video.
Las fotos quedan en `clasificador/vistazo/`. `MOSTRAR = True` las enseña en el cuaderno.

In [ ]:
MOSTRAR = True

xml = RAIZ / "botella.xml"
clasificador = cv2.CascadeClassifier(str(xml))
if clasificador.empty():
    raise SystemExit(f"No se pudo cargar {xml}")

def marcar(imagen):
    gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    cajas = clasificador.detectMultiScale(
        gris, scaleFactor=1.3, minNeighbors=8, minSize=(ANCHO, ALTO)
    )
    for (x, y, w, h) in cajas:
        cv2.rectangle(imagen, (x, y), (x + w, y + h), (70, 160, 70), 2)
        cv2.putText(
            imagen, "botella", (x, max(16, y - 6)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (70, 160, 70), 2, cv2.LINE_AA,
        )
    return imagen, len(cajas)

guardadas = []

captura = cv2.VideoCapture(str(VIDEO))
ok, frame = captura.read()
captura.release()
if ok:
    marcado, cantidad = marcar(frame)
    destino = VISTAZO / f"video_botellas_{cantidad}.jpg"
    cv2.imwrite(str(destino), marcado)
    guardadas.append(destino)
    print(f"Video: {cantidad} cajas -> {destino.name}")
else:
    print("No se pudo leer el video", VIDEO)

if MOSTRAR and guardadas:
    try:
        from IPython.display import Image, display
        for destino in guardadas:
            display(Image(filename=str(destino)))
    except Exception:
        for destino in guardadas:
            print("Abre la imagen:", destino)

## Compilar otro XML

Deja `COMPILAR = False` hasta tener `opencv_createsamples` y `opencv_traincascade` en el PATH.
`numPos` va por debajo del total de positivas, porque el entrenador descarta algunas.

El XML nuevo queda en `clasificador/salida/cascade.xml`. Esta celda no reemplaza `botella.xml`.

In [ ]:
COMPILAR = False

num_pos = max(0, int(len(lineas_info) * 0.85))
num_neg = len(negativas)

crea = shutil.which("opencv_createsamples") or shutil.which("opencv_createsamples.exe")
entrena = shutil.which("opencv_traincascade") or shutil.which("opencv_traincascade.exe")

comando_muestras = (
    f'"{crea or "opencv_createsamples"}" '
    f'-info "{INFO.relative_to(RAIZ).as_posix()}" -num {len(lineas_info)} '
    f'-w {ANCHO} -h {ALTO} -vec "{VECTOR.relative_to(RAIZ).as_posix()}"'
)
comando_cascade = (
    f'"{entrena or "opencv_traincascade"}" '
    f'-data "{SALIDA.relative_to(RAIZ).as_posix()}" '
    f'-vec "{VECTOR.relative_to(RAIZ).as_posix()}" '
    f'-bg "{FONDOS.relative_to(RAIZ).as_posix()}" '
    f'-numPos {num_pos} -numNeg {num_neg} -numStages {ETAPAS} '
    f'-w {ANCHO} -h {ALTO} -featureType HAAR -mode BASIC '
    f'-minHitRate 0.995 -maxFalseAlarmRate 0.5'
)

BAT.write_text(
    "\r\n".join([
        "@echo off",
        f'cd /d "{RAIZ}"',
        comando_muestras,
        "if errorlevel 1 exit /b 1",
        comando_cascade,
        "",
    ]),
    encoding="utf-8",
)

print(BAT.read_text(encoding="utf-8"))

if not COMPILAR:
    print("Compilación apagada. El clasificador que se usa ahora es botella.xml.")
elif not crea or not entrena:
    raise SystemExit(
        "No están opencv_createsamples ni opencv_traincascade en el PATH. "
        "Puedes lanzar clasificador/compilar.bat desde la carpeta donde sí estén."
    )
elif num_pos < 1 or num_neg < 1:
    raise SystemExit("No hay fotos suficientes.")
else:
    import subprocess
    subprocess.run(BAT, cwd=RAIZ, shell=True, check=True)
    print("Listo:", SALIDA / "cascade.xml")